In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import torch
from omegaconf import OmegaConf

from src.builder import build_critic, build_generator, build_image_loader, validate_config
from src.training.penalty import gradient_penalty
from src.training.trainer import _recon_loss, _slice_along_axis

In [2]:
cfg = OmegaConf.load(REPO_ROOT / "configs" / "default.yaml")
validate_config(cfg)

B = 2
C = cfg.data.in_channels
D, H, W = cfg.data.train_shape
axis = cfg.anchor.axis

dataset = build_image_loader(cfg)
netG = build_generator(cfg).eval()
netCs = [build_critic(cfg).eval() for _ in range(3)]

sparse = torch.zeros(B, C, D, H, W)
mask = torch.zeros(B, 1, D, H, W)
K = 3
imgs = dataset.sample(axis, B * K).view(B, K, C, H, W)
for b in range(B):
    for k, p in enumerate(torch.randperm(D)[:K].tolist()):
        sparse[b, :, p, :, :] = imgs[b, k]
        mask[b, :, p, :, :] = 1.0

with torch.no_grad():
    fake_3d = netG(sparse, mask)

print("fake_3d :", tuple(fake_3d.shape))

fake_3d : (2, 1, 64, 64, 64)


In [3]:
real = dataset.sample(axis, B * D)
fake = _slice_along_axis(fake_3d, axis)

with torch.no_grad():
    real_score = netCs[axis](real).mean()
    fake_score = netCs[axis](fake).mean()
gp = gradient_penalty(netCs[axis], real, fake, gp_lambda=cfg.trainer.gp_lambda)
critic_loss = fake_score - real_score + gp

print("real            :", tuple(real.shape))
print("fake            :", tuple(fake.shape))
print("real_score      :", real_score.item())
print("fake_score      :", fake_score.item())
print("gp              :", gp.item())
print("critic_loss     :", critic_loss.item())

real            : (128, 1, 64, 64)
fake            : (128, 1, 64, 64)
real_score      : 0.006999333389103413
fake_score      : 0.0002318722108611837
gp              : 9.535605430603027
critic_loss     : 9.528838157653809


In [4]:
fake_3d_g = netG(sparse, mask)
scores = [netCs[a](_slice_along_axis(fake_3d_g, a)).mean() for a in range(3)]
adv = -torch.stack(scores).mean()
rec = _recon_loss(fake_3d_g, sparse, mask)
gen_loss = adv + cfg.trainer.recon_lambda * rec

print("adv_loss   :", adv.item())
print("recon_loss :", rec.item())
print("gen_loss   :", gen_loss.item())

adv_loss   : -0.000162544209160842
recon_loss : 0.20734108984470367
gen_loss   : 2.0732483863830566
